### Astronomy Database Query
David Vance, Pima Community College  
Dr. Dennis Just, Pima Community College  
19 May 2026


ADQ takes advantage of the Application Programming Interface (API) from the European Space Agency (ESA) to access Gaia telescope information and uses database queries to access specific information.  This project directly builds off the **PIMA School (24 March 2024)** independant study project and utilizes the same environment build, with the addition of adding the **astroquery** library.  Instructions to add astroquery are included below.

The ESA's Gaia Project originally started as the GAIA Project: **Global Astrometric Interferometer of Astrophysics**, reflecting the original interferomy optical technique they planned to use, but over the course of the project's development that changed, and GAIA was renamed Gaia.

Launched 19 December 2013, Gaia conducted a Lissajous orbit in the Sun-Earth L2 Lagrangian point, its mission to measure positions, distances and motions of stars, and to calculate positions of exoplanets.  Gaia's mission statement directly supports this project's investigation into parallax and stellar distances.  This is the exact same orbital plan the James Webb telescope utilizes, as it takes minimal fuel to maintain its position.  

Gaia ran out of fuel in 2025, no longer able to maintain its L2 orbital position, and was moved into a safe heliocentric (or circumsolar) orbit and deactivated 27 March 2025.  All the data had been completely downloaded January 2025, with processing expected to continue until 2030.

Gaia is not unique in our ability to access data via API.  A list of other accessible projects will be given in the References for future investigations.  Projects may have modified or developed their own API; you will need to reference that project directly to determine how to access it.

### Installing astroquery

In addition to the elements we added to create the 'pima' environment in a prior AST296LB, we need to add astroquery.  

Open your terminal, move to your working directory, and activate your pima environment by entering in:

**conda activate pima**

Once that resolves, enter the following:

**python -m pip install -U --pre astroquery[all]**

This command will install astroquery, along with all its dependancies ([all] argument).


Or, you can also use conda to install:

  **conda install -c conda-forge astroquery**

### Import modules

In [2]:
# We need to initiate and import the appropriate modules in order to format and access all of our data.
%matplotlib inline

# Standard astronomy imports
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import sep

# Gaia-specific imports
from astroquery.gaia import Gaia

In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).


### OPTIONAL: Creating a Gaia Database account and logging in ####

You can register for an account at **https://www.cosmos.esa.int/web/gaia-users/register**.

It is not necessary to create an account and log into the Gaia database.  Creating and logging into an account will allow you to save tables and queries to the Gaia server for your projects, allowing you to share your work with others as well as using it for other research.  For what we are doing, you can access everything you need without this.

If you've created an account you can log in with the following command:

**Gaia.login()**

You will be prompted for your user name and password as we didn't pass any arguments.  You can use the below format to include passing your information if you feel comfortable doing so, but your system's password manager will also handle this for you.

**Gaia.login(user='userName', password='userPassword')**



In [3]:
#Log into Gaia with your COSMOS user account
Gaia.login()

INFO: Login to gaia TAP server [astroquery.gaia.core]


User:  dvance
Password:  ········


INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


### Accessing the Database and pulling table information

This project uses Gaia Data Release 3, which was the most recent data release at the start of this project.  Gaia Data Release 4 is scheduled for December of 2026 (GAia Data Release, 2026).

#### Load Tables

Our first action is to query the database for the database tables we can access.  We load the names of the tables in the Gaia database using the command:

**Gaia.load_tables(only_names=True)**

The option **only_names=True** returns just the information about the tables, not the actual contents.  We don't want to pull down the actual tables at this time, but only see which ones may have information we can use (Prusti, 2016).

    

In [2]:
tables = Gaia.load_tables(only_names=True)

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]


**And now we can diplay the table names:**

In [3]:
for table in tables:
    print(table.name)

external.apassdr9
external.catwise2020
external.gaiadr2_astrophysical_parameters
external.gaiadr2_geometric_distance
external.gaiaedr3_distance
external.gaiaedr3_gcns_main_1
external.gaiaedr3_gcns_rejected_1
external.gaiaedr3_spurious
external.gaia_eso_survey
external.galex_ais
external.lamost_dr9_lrs
external.lamost_dr9_mrs
external.ravedr5_com
external.ravedr5_dr5
external.ravedr5_gra
external.ravedr5_on
external.ravedr6
external.sdssdr13_photoprimary
external.skymapperdr1_master
external.skymapperdr2_master
external.tmass_xsc
external.xgboost_table1
external.xgboost_table2
gaiadr1.aux_qso_icrf2_match
gaiadr1.ext_phot_zero_point
gaiadr1.allwise_best_neighbour
gaiadr1.allwise_neighbourhood
gaiadr1.gsc23_best_neighbour
gaiadr1.gsc23_neighbourhood
gaiadr1.ppmxl_best_neighbour
gaiadr1.ppmxl_neighbourhood
gaiadr1.sdss_dr9_best_neighbour
gaiadr1.sdss_dr9_neighbourhood
gaiadr1.tmass_best_neighbour
gaiadr1.tmass_neighbourhood
gaiadr1.ucac4_best_neighbour
gaiadr1.ucac4_neighbourhood
gaiadr1.u

#### Accessing a table's meta data

This results in a lot of tables.  We can next query a specific table (**load_table**, not **load_tables**) and see its specific descriptive meta data.  For this project I am using **gaiadr3.gaia_source**.

In [4]:
meta = Gaia.load_table('gaiadr3.gaia_source')
meta

print(meta)

TAP Table name: gaiadr3.gaia_source
Description: This table has an entry for every Gaia observed source as published with this data release. It contains the basic source parameters, in their final state as processed by the Gaia Data Processing and Analysis Consortium from the raw data coming from the spacecraft. The table is complemented with others containing information specific to certain kinds of objects (e.g.~Solar--system objects, non--single stars, variables etc.) and value--added processing (e.g.~astrophysical parameters etc.). Further array data types (spectra, epoch measurements) are presented separately via Datalink resources.
Size (bytes): 3646930329600
Num. columns: 152


### Columns

Gaia3 has 152 columns of data.  The meta data is already loaded, so the following command will allow us to see every column in the database.

In [5]:
for column in meta.columns:
    print(column.name)

solution_id
designation
source_id
random_index
ref_epoch
ra
ra_error
dec
dec_error
parallax
parallax_error
parallax_over_error
pm
pmra
pmra_error
pmdec
pmdec_error
ra_dec_corr
ra_parallax_corr
ra_pmra_corr
ra_pmdec_corr
dec_parallax_corr
dec_pmra_corr
dec_pmdec_corr
parallax_pmra_corr
parallax_pmdec_corr
pmra_pmdec_corr
astrometric_n_obs_al
astrometric_n_obs_ac
astrometric_n_good_obs_al
astrometric_n_bad_obs_al
astrometric_gof_al
astrometric_chi2_al
astrometric_excess_noise
astrometric_excess_noise_sig
astrometric_params_solved
astrometric_primary_flag
nu_eff_used_in_astrometry
pseudocolour
pseudocolour_error
ra_pseudocolour_corr
dec_pseudocolour_corr
parallax_pseudocolour_corr
pmra_pseudocolour_corr
pmdec_pseudocolour_corr
astrometric_matched_transits
visibility_periods_used
astrometric_sigma5d_max
matched_transits
new_matched_transits
matched_transits_removed
ipd_gof_harmonic_amplitude
ipd_gof_harmonic_phase
ipd_frac_multi_peak
ipd_frac_odd_win
ruwe
scan_direction_strength_k1
scan_di

#### Specific Data Query

At this point we still have too much information to do anything meaningful.  What we can now do is use Astronomical Data Query Language (ADQL) to interact with the database for specific information.  The capitalized words are specific query commands:

* SELECT:  We are selecting data, not adding or modifying it
* TOP:     This limits us to only the first X rows of the table.  For this initial forray I only want to pull 10 values.
           We run into processing time concerns when querying for large numbers of entries.  When pulling 2000 or less entries, a synchroneous
           search (one right after the other) is acceptable, but anything more than that will result in significant time issues.  If we were to
           query for a larger sample, an asynchroneous search would be the way to go, meaning the process wouldn't wait for the initial database query
           to resolve completely before moving on to the next task, allowing us to start processing data while the entire search continues in the
           background.  This is also where you want to have a Gaia Database account, to save this to the server for your use.
* columns: We list the specific columns of data we want to pull from the database.
* FROM:    This defines which **table** we determined prior to look at.

The project's particular interest is parallax.  Looking at the column names I can see some interesting information I would like to get, and assign it to **query1**.  

In [21]:
query1 = """SELECT
            TOP 10
            source_id, designation, ra, dec, parallax, parallax_error, parallax_over_error
            FROM gaiadr3.gaia_source
         """

In order to run this query we use **launch_job**:

In [22]:
job = Gaia.launch_job(query1)

This creates an object named job containing the metadata as well a table of the information that we just requested.  Some key factors to note are that parallax is given in millearc-seconds, and that parallax error and parallax over error have been calculated for you.  Parallax over error is then evaluated to determine the 

In [23]:
print(job)

<Table length=10>
        name         dtype  unit                            description                            
------------------- ------- ---- ------------------------------------------------------------------
          source_id   int64      Unique source identifier (unique within a particular Data Release)
        designation  object             Unique source designation (unique across all Data Releases)
                 ra float64  deg                                                    Right ascension
                dec float64  deg                                                        Declination
           parallax float64  mas                                                           Parallax
     parallax_error float32  mas                                         Standard error of parallax
parallax_over_error float32                                  Parallax divided by its standard error
Jobid: None
Phase: COMPLETED
Owner: None
Output file: a6337df5-3a85-11f1-bd34-bc97

In [24]:
results = job.get_results()
results

source_id,designation,ra,dec,parallax,parallax_error,parallax_over_error
,,deg,deg,mas,mas,
int64,object,float64,float64,float64,float32,float32
2886218060160,Gaia DR3 2886218060160,45.156393315766216,0.14604926569549598,0.33489301828035556,0.78819346,0.42488682
4776005026176,Gaia DR3 4776005026176,44.88237668886769,0.1211487019722647,0.6584420987197187,1.7231603,0.3821131
5291399870976,Gaia DR3 5291399870976,44.92080406280173,0.14607416751960006,0.37984212153230634,0.06516887,5.828582
5394479088256,Gaia DR3 5394479088256,44.969991526820806,0.1635841165961584,1.8032767517717365,0.10560531,17.075626
5841155589888,Gaia DR3 5841155589888,44.840191120528694,0.1525615836147273,2.6425196658535244,0.24326652,10.862652
5944234902272,Gaia DR3 5944234902272,44.88476069792024,0.16480581643639847,-0.3956593009386258,1.340139,-0.2952375
6528350458496,Gaia DR3 6528350458496,44.907031901017376,0.20601107358897333,1.5085122963778241,0.09760174,15.455792
6700149148032,Gaia DR3 6700149148032,44.999423900148216,0.17945761649879463,0.00552615704400111,0.20559914,0.026878307


*At this point I want to start isolating the parallax data we have pulled to start calculating parallax.  I am also wanting to be able to use source_id to identify the specific object name we are looking at.  That is for next week!!*

#### Pulling information and doing calculations

Our results table is of the type:

In [25]:
type(results)

astropy.table.table.Table

What we are going to do is reduce the overall table into columns, which we can then access like we would a list.

In [26]:
source = results['source_id']
parallax = results['parallax']
new_designation = results['designation']
print(new_designation)

      designation      
-----------------------
 Gaia DR3 2886218060160
 Gaia DR3 4776005026176
 Gaia DR3 5291399870976
 Gaia DR3 5394479088256
 Gaia DR3 5841155589888
 Gaia DR3 5944234902272
 Gaia DR3 6528350458496
 Gaia DR3 6700149148032
 Gaia DR3 9281425163264
Gaia DR3 10136122991360


In [19]:
type(source)

astropy.table.column.MaskedColumn

Now we throw source and parallax into a loop.  Some values were given as '--', and some parallax were given as a negative.  Neither are of value to us, so the loop will only process values, and print a message if the values given are a negative.  Negative values happen when confusion occurs in the observation of a source, such as sources being too close together, too faint, or in a dense region.  There is value in negative parallax values (Luri et all, 2018).  As discussed in **REFLECTION** below, this is beyond me at this time, so I am choosing to eliminate negative values and simply address each source individually.

In [20]:
# Set accumulator to 0
index = 0

# Loop through the 10 values we have. Change this value if we download more records.
while index < 10:

    # Only calculate parallax that has values; skip if '--' is the value
    if parallax[index] != '--':
        pDistance = 1 / (parallax[index] * .001) # The parallax formula is in arc-seconds, but the information from Gaia is in milliarc-seconds
        lyDistance = pDistance * 3.26 # 1 parsec = 3.26 lightyears; calculate the distance in lightyears
        # distance_uncertainty = parallax_error[index] / (parallax[index] * .001) **2 # σd = σp/p^2.  Also convert milliarce-seconds to arc seconds
            
        if pDistance <= 0: # If parallax is a negative print an error message
            print(f'The parallax data for {source[index]} is uncertain; refer to Luri et all, 2018.')

        else: # Display the source's distance in parsec and lightyear.
            print(f'The distance to {source[index]} is {pDistance:.3f} parsecs or {lyDistance:.3f} lightyears')
            # print(f'Distance uncertainty is {distance_uncertainty}')
    index += 1

The distance to 343597448960 is 5098.200 parsecs or 16620.131 lightyears
The parallax data for 1099511693312 is uncertain; refer to Luri et all, 2018.
The distance to 5497558205696 is 1404.287 parsecs or 4577.976 lightyears
The distance to 6463926589568 is 6979.776 parsecs or 22754.069 lightyears
The distance to 7249904966144 is 6371.221 parsecs or 20770.181 lightyears
The distance to 8040178948096 is 755.635 parsecs or 2463.371 lightyears
The distance to 8315056861440 is 27935.467 parsecs or 91069.623 lightyears
The distance to 8383776596736 is 2534.165 parsecs or 8261.377 lightyears
The distance to 9109626472192 is 3333.954 parsecs or 10868.690 lightyears


### REFLECTION ###
The Parallax Project started out by reading Stellar Parallax in An Introduction to Modern Astrophysics (Ostlie & Carroll, 1996).  Parallax is the apparent movement of a distant object against a backdrop of even more distant objects when seen from two different locations.  When observing from Earth, we use the Earth's orbital movement around the sun, allowing us to observe from one side of the sun and then observe from the other side six months later, giving us a full 2 AU distance from side to side.  This establishes our baseline and the parallax angle (one-half of the observed shift from side to side), allowing us to use trigonometry to calculate the distance to the observed stellar object.

That turns out to be a simplified answer.  One of the basic problems with parallax is that the apparent movement is incredibly small, requiring precise instrumentation.  Even then, we're limited to determining the distances to only our closest neighbors.  The angles created by Earth's rotation around the sun to more distant objects become similar to 0", meaning we don't have enough angle to calculate parallax.

A technical demonstration was conducted in 2020 with the New Horizons spacecraft allowed us to see the apparent shift to Proxima Centauri and Wolf 359 (NASA, 2023).  New Horizons was located more than 4.3 billion miles from Earth, or over 43 AU distant, giving us a much larger angle to observe the two mentioned stars.  This was only a technical demonstration due to lack of precision regarding New Horizon's long-range telescopic camera.

An interesting sidenote concerning the New Horizons project is that one of the collaborating astrophysicists is Dr. Brian May, guitarist for Queen.

While I had the math for the project, I needed to find the parallax values.  Dr. Dennis Just mentored me to look at the European Space Agency's (ESA) Gaia program.  

Gaia was a space observatory platform orbiting in Earth's L2 Lagrange point.  Launched in 2013, Gaia completed its observational mission 15 January 2025, and in March it was moved into a continual solar orbit, where it will remain unless something external acts upon it.  Gaia currently has 3 published Data Releases, with Data Release 4 expected in December of this year, and Data Release 5, the final release of all data, some time after the end of 2030.  This project specifically uses Data Release 3.  However, with Gaia's DR4 upcoming, there may be some instabilities working with the Gaia database.  For example, theree was about a week-long disruption end of April, preventing database access while maintenence was being conducted.

I used Astronomical Data Query Language (ADQL) to access the Gaia database (Gaia Users, 2026) .  ADQL is an astronomy-focused dialect of Structured Query Language (SQL), allowing us to access the Gaia and other astronomy databases.  Not all astronomy projects follow AQDL; you will need to access the specific project to see what they use to access data.  Using various Python commands I was able to extract specific table data, build a database query, and to extract parallax data for computation.

An important consideration is that the parallax formula is in arcseconds, whereas the Gaia database information is in milliarcseconds.  A scale conversion is necessary.

I discussed this as being a simplified answer regarding parallax.  In our research into parallax we discovered concerns, such as parallax bias, parallax error, and negative parallax.  While not cited here, I will provide sources in the Resources segment.  Right now, the math involved, specifically Baysian statistics and probablity analysisused to incorporate negative parallax data is beyond me.  

Overall, the Parallax Project taught what parallax was, how to explain it, how to access database information from observatories and other projects, how to calculate parallax and present it in a meaningful manner, resulting in tangible project that others can use.  I thought this was an outstanding learning opportunity, and I want to be able to do more meaningful database access in the future.

### REFERENCES

Prusti, T., et al. “The Gaia Mission.” Astronomy & Astrophysics, EDP Sciences, 25 Nov. 2016, www.aanda.org/articles/aa/full_html/2016/11/aa29272-16/aa29272-16.html. 

Ostlie, Dale A., and Bradley W. Carroll. An Introduction to Modern Stellar Astrophysics. Addison-Wesley, 1996. 

“NASA’s New Horizons Conducts the First Interstellar Parallax Experiment.” NASA, NASA, 29 Sept. 2023, www.nasa.gov/solar-system/nasas-new-horizons-conducts-the-first-interstellar-parallax-experiment/. 

“How to Write ADQL Queries for Gaia Data - Gaia Users - Cosmos.” Gaia Users, www.cosmos.esa.int/web/gaia-users/archive/writing-queries. Accessed 19 May 2026. 

Thomas Bayes | Bayesian Statistics, Probability Theory, Logic | Britannica, www.britannica.com/biography/Thomas-Bayes. Accessed 15 Nov. 2025. 

Luri, X., et al. “Gaia Data Release 2: Using Gaia Parallaxes.” arXiv.Org, 8 June 2018, arxiv.org/abs/1804.09376. 

Estimating Distances from Parallaxes Coryn A.L. Bailer-Jones, bailer-jones.www3.mpia.de/parallax.pdf. Accessed 19 May 2026. 

“Gaia Data Release Scenario - Gaia - Cosmos.” Gaia, www.cosmos.esa.int/web/gaia/release#:~:text=On%20this%20webpage%2C%20the%20target%20release%20scenario%20is,will%20be%20released%20only%20with%20the%20final%20catalogue. Accessed 19 May 2026. 

